In [ ]:
import ee
import geemap
import geopandas

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(
    project='my-project-01-487803',  # replace with your GEE project ID
    opt_url='https://earthengine.googleapis.com'
)

In [ ]:
m = geemap.Map()
m

In [ ]:
bounds = '/content/wnc_zips_bounds.geojson'
fc = geemap.geojson_to_ee(bounds)
geometry = fc.geometry()
m.add_layer(fc.style(**{'color': 'ff0000', 'fillColor': '00000000', 'width': 1}), {}, 'Zip Codes')

In [ ]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

preHelene = ee.Date('2024-09-24')
postHelene = ee.Date('2024-10-04')

In [ ]:
def cloud_mask(img):
    qaBand = 'cs_cdf'
    clearThreshold = 0.5
    return img.updateMask(img.select(qaBand).gte(clearThreshold))

csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

In [ ]:
qaBand = 'cs_cdf'

preComposite = (
    s2
    .filter(ee.Filter.date(preHelene.advance(-1, 'month'), preHelene))
    .filterBounds(geometry)
    .linkCollection(csPlus, [qaBand])
    .map(cloud_mask)
    .median()
)

In [ ]:
postComposite = (
    s2
    .filter(ee.Filter.date(postHelene, postHelene.advance(1, 'month')))
    .filterBounds(geometry)
    .linkCollection(csPlus, [qaBand])
    .map(cloud_mask)
    .median()
)

In [ ]:
def addNDVI(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('ndvi')
    return image.addBands(ndvi)

In [ ]:
preNDVI = addNDVI(preComposite).select('ndvi')
postNDVI = addNDVI(postComposite).select('ndvi')

In [ ]:
ndviVis = {
    'min': -0.4,
    'max': 0.9,
    'palette': ['red', 'yellow', 'orange', 'green']
}

m.add_layer(preNDVI.clip(geometry), ndviVis, 'Pre-Helene NDVI')
m.add_layer(postNDVI.clip(geometry), ndviVis, 'Post-Helene NDVI')

In [ ]:
pre_export = ee.batch.Export.image.toDrive(
    image=preNDVI.clip(geometry),
    description='preHelene_NDVI',
    folder='GEE_Exports',
    fileNamePrefix='preHelene_NDVI',
    scale=10,
    region=geometry,
    maxPixels=1e13,  # increase from 1e9 to 1e13
    fileFormat='GeoTIFF'
)

post_export = ee.batch.Export.image.toDrive(
    image=postNDVI.clip(geometry),
    description='postHelene_NDVI',
    folder='GEE_Exports',
    fileNamePrefix='postHelene_NDVI',
    scale=10,
    region=geometry,
    maxPixels=1e13,  # increase from 1e9 to 1e13
    fileFormat='GeoTIFF'
)

pre_export.start()
post_export.start()
print('Exports started!')